### **Submissão 3-A — Ensemble (Weighted Voting)**

**Grupo 1 · MIA · Aprendizagem Profunda**

**▶ CORRER DEPOIS** do Ensemble.ipynb (precisa do `ensemble-config.json` e `support-set.csv`)

Corre os 3 LLMs com **checkpoints por modelo** e aplica weighted voting.

- Cada modelo guarda as previsões em `subm3-preds-{modelo}.json`
- Se já existir checkpoint, **carrega e salta** (zero chamadas API)
- O Notebook B carrega estes mesmos checkpoints

Output: `subm3-g1-MIA-A.csv`

In [ ]:
import pandas as pd
import numpy as np
import time
import os
import json
from collections import Counter
from tqdm.auto import tqdm

LABELS = ['Anthropic', 'Google', 'Human', 'Meta', 'OpenAI']

### **1. Carregar configuração do Ensemble.ipynb**

In [ ]:
with open('../Subm3/ensemble-config.json', 'r') as f:
    config = json.load(f)

weights   = config['weights']
best_solo = config['best_solo_model']

print('📊 Configuração carregada:')
print(f'   Esquema pesos: {config["best_scheme"]}')
print(f'   Ensemble acc (val): {config["ensemble_accuracy"]:.2%}')
print(f'   Melhor solo: {best_solo} ({config["best_solo_accuracy"]:.2%})')
print(f'\nPesos:')
for name, w in sorted(weights.items(), key=lambda x: -x[1]):
    acc = config['models'][name]['accuracy']
    print(f'   {name}: {w:.3f} (acc val={acc:.2%})')

df_support = pd.read_csv('../Subm3/support-set.csv', sep=';')
df_support.columns = df_support.columns.str.strip().str.lower()
print(f'\nSupport set: {len(df_support)} exemplos')

### **2. Carregar dataset de submissão**

In [ ]:
df_subm = pd.read_csv('../database/dataset-subm3.csv', sep=';')
df_subm.columns = df_subm.columns.str.strip().str.lower()
print(f'Submissão 3: {len(df_subm)} textos')
print(f'IDs: {df_subm["id"].iloc[0]} ... {df_subm["id"].iloc[-1]}')

### **3. APIs e funções**

In [ ]:
import anthropic
from google import genai
from openai import OpenAI

# API KEYS
ANTHROPIC_KEY = 'sk-ant-...'
GOOGLE_KEY    = '...'
DEEPSEEK_KEY  = 'sk-...'

claude_client   = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
gemini_client   = genai.Client(api_key=GOOGLE_KEY)
deepseek_client = OpenAI(api_key=DEEPSEEK_KEY, base_url='https://api.deepseek.com')

CLAUDE_MODEL   = config['models']['claude']['model_id']
GEMINI_MODEL   = config['models']['gemini']['model_id']
DEEPSEEK_MODEL = config['models']['deepseek']['model_id']

print(f'Modelos: {CLAUDE_MODEL}, {GEMINI_MODEL}, {DEEPSEEK_MODEL}')

In [ ]:
def ask_claude(prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            resp = claude_client.messages.create(
                model=CLAUDE_MODEL, max_tokens=10, temperature=0.0,
                messages=[{'role': 'user', 'content': prompt}]
            )
            return resp.content[0].text.strip()
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 5 * (attempt + 1)
                print(f'    ⚠️ Claude retry {attempt+1} ({wait}s): {e}')
                time.sleep(wait)
            else:
                return f'Error: {e}'

def ask_gemini(prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            resp = gemini_client.models.generate_content(
                model=GEMINI_MODEL, contents=prompt,
                config={'temperature': 0.0, 'max_output_tokens': 10}
            )
            return resp.text.strip()
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 5 * (attempt + 1)
                print(f'    ⚠️ Gemini retry {attempt+1} ({wait}s): {e}')
                time.sleep(wait)
            else:
                return f'Error: {e}'

def ask_deepseek(prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            resp = deepseek_client.chat.completions.create(
                model=DEEPSEEK_MODEL, temperature=0.0, max_tokens=10,
                messages=[{'role': 'user', 'content': prompt}]
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 5 * (attempt + 1)
                print(f'    ⚠️ DeepSeek retry {attempt+1} ({wait}s): {e}')
                time.sleep(wait)
            else:
                return f'Error: {e}'

LLM_FUNCTIONS = {
    'claude':   ask_claude,
    'gemini':   ask_gemini,
    'deepseek': ask_deepseek,
}

# Teste rápido
for name, fn in LLM_FUNCTIONS.items():
    print(f'  {name}: {fn("Say only: hello")}')

In [ ]:
def build_few_shot_prompt(text, support_examples):
    examples_block = ''
    for _, row in support_examples.iterrows():
        ex_text = row['text'][:400]
        examples_block += f'Text: {ex_text}\nCategory: {row["label"]}\n\n'

    return f"""You are an expert at detecting AI-generated text. Classify the following text into exactly ONE of these categories:

- Human (written by a human, e.g. from Wikipedia)
- Anthropic (generated by Claude)
- Google (generated by Gemini)
- Meta (generated by Llama)
- OpenAI (generated by GPT)

Here are labeled examples:

{examples_block}
Now classify this text. Output ONLY the category name.

Text: {text}
Category:"""

def normalize_prediction(raw):
    if not raw or raw.startswith('Error'):
        return None
    raw_lower = raw.strip().lower()
    for label in LABELS:
        if label.lower() in raw_lower:
            return label
    return None

def classify_texts(texts, support_df, llm_fn, sleep_sec=1.0):
    predictions = []
    for text in tqdm(texts, desc='Classificando'):
        prompt = build_few_shot_prompt(text, support_df)
        raw = llm_fn(prompt)
        pred = normalize_prediction(raw)
        predictions.append(pred)
        if sleep_sec > 0:
            time.sleep(sleep_sec)
    return predictions

def weighted_vote(predictions_per_model, weights, fallback='Human'):
    n = len(next(iter(predictions_per_model.values())))
    final = []
    for i in range(n):
        votes = Counter()
        for model_name, preds in predictions_per_model.items():
            pred = preds[i]
            if pred is not None:
                votes[pred] += weights[model_name]
        final.append(votes.most_common(1)[0][0] if votes else fallback)
    return final

### **4. Classificar com os 3 LLMs**

Cada modelo guarda `subm3-preds-{modelo}.json` logo ao acabar.
Se o checkpoint já existir, carrega e salta — zero chamadas API.

In [ ]:
subm_texts = df_subm['text'].tolist()
subm_ids   = df_subm['id'].tolist()
os.makedirs('../Subm3', exist_ok=True)
subm_preds = {}  # {model_name: {id_str: pred}}

for model_name, llm_fn in LLM_FUNCTIONS.items():
    ckpt_path = f'../Subm3/subm3-preds-{model_name}.json'

    # ── Carregar checkpoint existente (dict id → pred) ──
    existing = {}
    if os.path.exists(ckpt_path):
        with open(ckpt_path, 'r') as f:
            data = json.load(f)
            # Compatibilidade: converter lista antiga para dict
            if isinstance(data, list):
                existing = {str(subm_ids[i]): p for i, p in enumerate(data)}
            else:
                existing = data

    # ── Identificar textos que faltam ──
    missing_idx = [i for i, tid in enumerate(subm_ids) if str(tid) not in existing]

    if len(missing_idx) == 0:
        n_valid = sum(1 for v in existing.values() if v is not None)
        print(f'⏩ {model_name.upper()} — checkpoint completo ({n_valid}/{len(existing)} válidas), a saltar.')
    else:
        if existing:
            print(f'🔄 {model_name.upper()} — {len(existing)} em cache, faltam {len(missing_idx)}')
        else:
            print(f'\n{"═" * 50}')
            print(f'🔄 {model_name.upper()} — {len(subm_texts)} textos')
            print(f'{"═" * 50}')

        # ── Classificar um a um com checkpoint incremental ──
        for count, i in enumerate(tqdm(missing_idx, desc=f'{model_name}'), 1):
            prompt = build_few_shot_prompt(subm_texts[i], df_support)
            raw = llm_fn(prompt)
            pred = normalize_prediction(raw)
            existing[str(subm_ids[i])] = pred

            # Guardar após cada texto
            with open(ckpt_path, 'w') as f:
                json.dump(existing, f)

            time.sleep(1.0)

        valid = sum(1 for v in existing.values() if v is not None)
        dist = Counter(v for v in existing.values() if v)
        print(f'  ✅ {valid}/{len(existing)} válidas — {dict(dist)}')
        print(f'  💾 Checkpoint guardado em {ckpt_path}')

    # Reconstruir lista na ordem do dataset
    subm_preds[model_name] = [existing.get(str(tid)) for tid in subm_ids]

# Guardar também o consolidado
with open('../Subm3/subm3-raw-preds.json', 'w') as f:
    json.dump({k: [existing.get(str(tid)) for tid in subm_ids] for k in subm_preds}, f)
print('\n💾 Previsões consolidadas guardadas')


### **5. Exportar submissão A — Ensemble (Weighted Voting)**

In [ ]:
ids = df_subm['id'].tolist()

labels_a = weighted_vote(subm_preds, weights)

df_a = pd.DataFrame({'ID': ids, 'Label': labels_a})
path_a = '../Subm3/subm3-g1-MIA-A.csv'
df_a.to_csv(path_a, sep=';', index=False, encoding='utf-8')

print(f'{"═" * 50}')
print(f'SUBMISSÃO A — ENSEMBLE ({config["best_scheme"]})')
print(f'Acc validação: {config["ensemble_accuracy"]:.2%}')
print(f'Pesos: {", ".join(f"{k}: {v:.3f}" for k, v in sorted(weights.items(), key=lambda x: -x[1]))}')
print(f'{"═" * 50}')
print(df_a['Label'].value_counts().to_string())
print(f'\n✅ {path_a} ({len(df_a)} linhas)')

### **6. Validação**

In [ ]:
assert len(df_a) == len(df_subm), 'Subm A: tamanho errado'
assert list(df_a.columns) == ['ID', 'Label'], 'Subm A: colunas erradas'
assert all(l in LABELS for l in df_a['Label']), 'Subm A: labels inválidos'
print(f'✅ Subm A OK: {len(df_a)} linhas, labels válidos')
print(f'\n🎯 Ficheiro: {path_a}')